In [1]:
import openai 
import instructor 
from qdrant_client import QdrantClient
from pydantic import BaseModel, Field

In [2]:
prompt = """
You are a helful assistant.
Return an answer to the question.
Question: What is the capital of France?
"""

In [5]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


In [19]:
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
   temperature=0.7
)

In [20]:
print(response)

ChatCompletion(id='chatcmpl-Dg4YAZ6SFRSoh4ABPQqSDGlvWjR1v', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1778919638, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_3643be3370', usage=CompletionUsage(completion_tokens=7, prompt_tokens=31, total_tokens=38, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [21]:
print(response.choices[0].message.content)

The capital of France is Paris.


Add Instructor for Structured Outputs

In [22]:
client = instructor.from_openai(openai.OpenAI())

In [23]:
client

In [24]:
class RagGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    response_model=RagGenerationResponse,
    temperature=0.3
)
print(response)

answer='The capital of France is Paris.'


In [25]:
response = client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    response_model=RagGenerationResponse,
    temperature=0.3
)
print(response)

(RagGenerationResponse(answer='The capital of France is Paris.'), ChatCompletion(id='chatcmpl-Dg4YGjSSPqAcBWHw6AJKWusFcsRFT', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7qWfb9vmIGfQ4teXT6n2m3lT', function=Function(arguments='{"answer":"The capital of France is Paris."}', name='RagGenerationResponse'), type='function')]))], created=1778919644, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_84b19afcfe', usage=CompletionUsage(completion_tokens=11, prompt_tokens=97, total_tokens=108, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))))


In [26]:
class RagGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    reasoning: str = Field(description="The reasoning behind the answer")
response = client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    response_model=RagGenerationResponse,
    temperature=0.3
)
print(response)

(RagGenerationResponse(answer='The capital of France is Paris.', reasoning='Paris is internationally recognized as the capital city of France. It is the largest city in the country and serves as the political, cultural, and economic center of France.'), ChatCompletion(id='chatcmpl-Dg4a7bdW4ZgSlkThEOVXJ9RFtI4ui', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_27btVCq79dvqJYzGEjdSrPXa', function=Function(arguments='{"answer":"The capital of France is Paris.","reasoning":"Paris is internationally recognized as the capital city of France. It is the largest city in the country and serves as the political, cultural, and economic center of France."}', name='RagGenerationResponse'), type='function')]))], created=1778919759, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='def

Rag Example

In [28]:
class RagGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    reasoning: str = Field(description="The reasoning behind the answer")

In [29]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):   
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="amazon_reviews_collection",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    retrieve_context_details = []
    retrieve_context_product_names = []
    retrieve_context_helpful_votes = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(payload.get('product_id'))
        retrieve_context.append(payload.get('review_text', ''))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(payload.get('rating', 0))
        retrieve_context_helpful_votes.append(payload.get('helpful_votes', 0))
        retrieve_context_product_names.append(payload.get('product_name', ''))
        retrieve_context_details.append(payload.get('details', ''))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_ratings': retrieved_context_ratings,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_product_names': retrieve_context_product_names,
        'retrieve_context_helpful_votes': retrieve_context_helpful_votes
    }


def process_context(context):
    formatted_context = []
    for id, chunk, rating, details, product_name, helpful_votes in zip(context['retrieved_context_ids'], context['retrieve_context'], context['retrieved_context_ratings'], context['retrieve_context_details'], context['retrieve_context_product_names'], context['retrieve_context_helpful_votes']):
        formatted_context.append(f"ID: {id}\nRating: {rating}\nReview: {chunk}\nDetails: {details}\nProduct Name: {product_name}\nHelpful Votes: {helpful_votes}\n---")
    return "\n".join(formatted_context)


def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      {preprocessed_context}
      Question: {question}
      Answer:"""
    return prompt


def gen_answer(prompt):
    # Call may return different shapes depending on client used (OpenAI SDK or a helper
    # that returns a Pydantic model). Handle both cases and normalize to a dict.
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_model=RagGenerationResponse
    )

    return response, raw_response
   


def rag_pipeline(question, qdrant_client, top_k=5):
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    gen, raw_gen = gen_answer(prompt)

    # Normalize final answer whether gen is dict-like or an object
    if isinstance(gen, dict):
        answer_text = gen.get("text") or gen.get("answer") or str(gen)
        rag_generation_response = gen
    else:
        answer_text = getattr(gen, "answer", getattr(gen, "text", str(gen)))
        # try to convert to dict if pydantic object provides model_dump
        try:
            rag_generation_response = gen.model_dump() if hasattr(gen, "model_dump") else gen.dict()
        except Exception:
            rag_generation_response = str(gen)

    final_result = {
        "question": question,
        "answer": answer_text.answer if isinstance(answer_text, RagGenerationResponse) else answer_text,
        "retrieved_context": retrieve_context,
        "rag_generation_response": rag_generation_response,
    }

    return final_result


In [30]:
rag_pipeline("What do the reviews say about the durability of the product?", qdrant_client=QdrantClient(url=qdrant_url, api_key=qdrant_api_key), top_k=5)

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_45797/688307389.py:1: UserWarning: Api key is used with an insecure connection.
  rag_pipeline("What do the reviews say about the durability of the product?", qdrant_client=QdrantClient(url=qdrant_url, api_key=qdrant_api_key), top_k=5)


{'question': 'What do the reviews say about the durability of the product?',
 'answer': 'The reviews mention that the products have "excellent build quality," which implies good durability. However, there is no explicit mention of durability in the reviews.',
 'retrieved_context': {'retrieved_context_ids': ['B010',
   'B007',
   'B007',
   'B010',
   'B009'],
  'retrieve_context': ['This product has excellent build quality. great product.',
   'This product has excellent build quality. worth buying.',
   'This product has excellent build quality. worth buying.',
   'This product has excellent build quality. worth buying.',
   'This product has excellent build quality. pretty decent.'],
  'similarity_scores': [0.41704765,
   0.4083048,
   0.4082937,
   0.4082937,
   0.39687562],
  'retrieved_context_ratings': [4, 5, 3, 4, 1],
  'retrieve_context_details': [{'Date First Available': 'November 16, 2024',
    'Manufacturer': 'Brand D',
    'ASIN': 'ASIN4488260'},
   {'Date First Available':